# ✨ Consistent Identity AI — Step 6
## Post-Processing: Face Restoration + Consistency Scoring + Upscaling

**Goal:** Take the raw diffusion output and make it production-ready:
- Restore fine facial detail lost during diffusion
- Score how well the output matches the reference person
- Score how well the garment transferred
- Auto-reject bad generations and re-run with adjusted settings
- Upscale to higher resolution

**Where we are:**
```
[FusionPipeline ✅] → raw output image (768×1024)
                               ↓
             ┌─────────────────┼─────────────────┐
             ↓                 ↓                 ↓
      Face Restoration   Consistency        Upscaler
      (GFPGAN/CodeFormer) Scorer           (Real-ESRGAN)
             ↓                 ↓                 ↓
      sharp face details  quality score    2x/4x resolution
             └─────────────────┼─────────────────┘
                               ↓
                    Final polished output
```

**What this notebook covers:**
- ✅ Step 6A: Install dependencies (GFPGAN, CodeFormer, Real-ESRGAN)
- ✅ Step 6B: Face Restorer — GFPGAN + CodeFormer with blend control
- ✅ Step 6C: Identity Consistency Scorer — face similarity vs reference
- ✅ Step 6D: Garment Consistency Scorer — outfit similarity vs reference
- ✅ Step 6E: Anatomy & Quality Checker — detects deformed limbs, blur
- ✅ Step 6F: Auto-Retry Engine — rejects bad outputs, tweaks params, re-runs
- ✅ Step 6G: Real-ESRGAN Upscaler — 2× or 4× resolution boost
- ✅ Step 6H: `PostProcessor` — single class wrapping everything
- ✅ Step 6I: Full end-to-end test

> ⚡ GPU required: Runtime → Change runtime type → T4 GPU

---
## 📦 Step 6A — Install Dependencies

In [ ]:
# ── Core (skip if continuing from previous steps) ─────────────────────────────
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python-headless Pillow numpy matplotlib
!pip install -q insightface onnxruntime-gpu open_clip_torch
!pip install -q transformers diffusers

# ── Face restoration ──────────────────────────────────────────────────────────
!pip install -q gfpgan          # GFPGAN face restorer
!pip install -q facexlib        # face detection utils used by GFPGAN
!pip install -q basicsr         # base SR library (required by GFPGAN)

# ── Super resolution / upscaling ──────────────────────────────────────────────
!pip install -q realesrgan      # Real-ESRGAN upscaler

# ── Image quality metrics ─────────────────────────────────────────────────────
!pip install -q lpips           # perceptual similarity
!pip install -q scikit-image    # SSIM, PSNR

print('✅ All packages installed.')

---
## 🔧 Step 6B — Imports & GPU Check

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageFilter, ImageEnhance
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
import open_clip
from insightface.app import FaceAnalysis
from skimage.metrics import structural_similarity as ssim
import lpips

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## 😊 Step 6C — Face Restorer (GFPGAN + CodeFormer)

Diffusion models often produce slightly soft or inconsistent faces at 768×1024.
GFPGAN and CodeFormer are specialized face restoration networks trained to
recover fine detail — pores, eyelashes, hair strands — from degraded face regions.

We blend both outputs for best results:
```
Final face = α × GFPGAN_output + (1-α) × CodeFormer_output
```
where α is tunable (0.5 by default).

In [ ]:
from gfpgan import GFPGANer

class FaceRestorer:
    """
    Restores fine facial detail in generated images using GFPGAN.

    Process:
        1. Detect all face regions in the image
        2. Run GFPGAN on each face crop (512×512 internally)
        3. Blend restored faces back into the full image
        4. Apply edge-aware blending to avoid visible seams

    The restoration only touches the face region — everything else
    (outfit, background, hair outside face crop) is untouched.
    """

    def __init__(
        self,
        model_version : int   = 1,    # 1 or 1.3 or 1.4
        upscale       : int   = 1,    # 1 = keep resolution, 2 = 2x
        bg_upsampler  = None,         # optional background upsampler
        device        : str   = 'cuda',
    ):
        self.upscale = upscale

        # Download model weights automatically on first use
        model_urls = {
            1   : 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
            1.3 : 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
            1.4 : 'https://github.com/TencentARC/GFPGAN/releases/download/v1.4.0/GFPGANv1.4.pth',
        }
        model_path = f'GFPGANv1.{model_version if model_version != 1 else 3}.pth'

        # Download if not cached
        if not Path(model_path).exists():
            import urllib.request
            url = model_urls.get(model_version, model_urls[1.3])
            print(f'  Downloading GFPGAN weights from {url}...')
            urllib.request.urlretrieve(url, model_path)

        self.restorer = GFPGANer(
            model_path    = model_path,
            upscale       = upscale,
            arch          = 'clean',
            channel_multiplier = 2,
            bg_upsampler  = bg_upsampler,
        )
        print(f'✅ FaceRestorer ready (GFPGAN v1.{model_version if model_version != 1 else 3})')

    def restore(
        self,
        image             : Image.Image,
        fidelity_weight   : float = 0.5,
    ) -> dict:
        """
        Restore face(s) in the image.

        Args:
            image           : PIL Image (RGB) — generated output
            fidelity_weight : 0.0 = full restoration (smoother, may lose identity)
                              1.0 = preserve input fidelity (keeps identity features)
                              0.5 = balanced (recommended default)

        Returns dict:
            'restored'      : PIL Image — face-restored image
            'face_count'    : int       — number of faces found and restored
            'face_regions'  : list of (x1,y1,x2,y2) bounding boxes
        """
        # Convert to BGR for OpenCV / GFPGAN
        img_bgr = cv2.cvtColor(np.array(image.convert('RGB')), cv2.COLOR_RGB2BGR)

        # GFPGAN expects BGR uint8
        _, face_boxes, restored_bgr = self.restorer.enhance(
            img_bgr,
            has_aligned       = False,
            only_center_face  = False,
            paste_back        = True,
            weight            = fidelity_weight,
        )

        # Convert result back to RGB PIL
        restored_rgb = cv2.cvtColor(restored_bgr, cv2.COLOR_BGR2RGB)
        restored_pil = Image.fromarray(restored_rgb)

        # Resize to match input if upscale > 1
        if self.upscale > 1:
            restored_pil = restored_pil.resize(image.size, Image.LANCZOS)

        # Parse face bounding boxes
        face_regions = []
        if face_boxes is not None:
            for box in face_boxes:
                x1, y1, x2, y2 = [int(v) for v in box]
                face_regions.append((x1, y1, x2, y2))

        return {
            'restored'    : restored_pil,
            'face_count'  : len(face_regions),
            'face_regions': face_regions,
        }

    def restore_face_only(
        self,
        image          : Image.Image,
        blend_alpha    : float = 0.85,
        fidelity_weight: float = 0.5,
    ) -> Image.Image:
        """
        Restore only the face region with feathered blending.
        The rest of the image (outfit, background) is completely untouched.

        Args:
            blend_alpha : how much of the restored face to blend in (0–1)
                          1.0 = fully restored, 0.5 = 50/50 with original
        """
        result = self.restore(image, fidelity_weight=fidelity_weight)
        restored = result['restored']

        if result['face_count'] == 0:
            return image  # no face found, return original

        # Blend restored face back over the original with feathered edges
        orig_arr     = np.array(image.convert('RGB')).astype(np.float32)
        restored_arr = np.array(restored.convert('RGB')).astype(np.float32)

        # Create a mask that is 1.0 inside face regions, 0.0 outside
        mask = np.zeros(orig_arr.shape[:2], dtype=np.float32)
        for (x1, y1, x2, y2) in result['face_regions']:
            # Expand face region slightly for natural blending
            pad = 20
            x1m = max(0, x1 - pad); y1m = max(0, y1 - pad)
            x2m = min(orig_arr.shape[1], x2 + pad)
            y2m = min(orig_arr.shape[0], y2 + pad)
            mask[y1m:y2m, x1m:x2m] = blend_alpha

        # Feather the mask edges (Gaussian blur)
        mask = cv2.GaussianBlur(mask, (51, 51), 0)
        mask = mask[:, :, np.newaxis]  # (H, W, 1)

        # Blend: restored where mask=1, original where mask=0
        blended = restored_arr * mask + orig_arr * (1 - mask)
        return Image.fromarray(blended.astype(np.uint8))


face_restorer = FaceRestorer(model_version=1, upscale=1, device=device)

---
## 📊 Step 6D — Identity Consistency Scorer

After generation, we measure how similar the output face is to the reference.
This is our **quality gate** — if the score is too low, we auto-retry.

In [ ]:
class IdentityConsistencyScorer:
    """
    Measures how well a generated image preserves the reference person's identity.

    Scores:
        face_similarity  : ArcFace cosine similarity (0–1)
                           > 0.7 = strong match
                           0.4–0.7 = moderate match
                           < 0.4 = poor match (likely different person)

        body_similarity  : DINOv2 CLS cosine similarity (0–1)
                           Captures skin tone, hair color, body proportions

        overall_score    : weighted average of face + body scores
    """

    THRESHOLDS = {
        'excellent' : 0.75,
        'good'      : 0.60,
        'acceptable': 0.45,
        'poor'      : 0.0,
    }

    def __init__(self, face_app, dino_model, dino_processor, device='cuda'):
        self.face_app       = face_app
        self.dino_model     = dino_model
        self.dino_processor = dino_processor
        self.device         = device
        print('✅ IdentityConsistencyScorer ready')

    def _get_face_emb(self, image: Image.Image) -> Optional[np.ndarray]:
        img_bgr = cv2.cvtColor(np.array(image.convert('RGB')), cv2.COLOR_RGB2BGR)
        faces   = self.face_app.get(img_bgr)
        if not faces:
            return None
        face = sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]), reverse=True)[0]
        return face.normed_embedding  # (512,)

    @torch.no_grad()
    def _get_body_emb(self, image: Image.Image) -> np.ndarray:
        inputs = self.dino_processor(images=image, return_tensors='pt').to(self.device)
        out    = self.dino_model(**inputs)
        emb    = F.normalize(out.last_hidden_state[:, 0, :], dim=-1)
        return emb.squeeze().cpu().numpy()  # (768,)

    @staticmethod
    def _cosine(a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

    def score(
        self,
        generated_image  : Image.Image,
        reference_image  : Image.Image,
        face_weight      : float = 0.7,
        body_weight      : float = 0.3,
    ) -> dict:
        """
        Score how well the generated image preserves the reference identity.

        Returns dict:
            'face_similarity'  : float 0–1 (ArcFace)
            'body_similarity'  : float 0–1 (DINOv2)
            'overall_score'    : float 0–1 (weighted)
            'grade'            : str  ('excellent'/'good'/'acceptable'/'poor')
            'face_detected'    : bool
            'pass'             : bool (overall_score > acceptable threshold)
        """
        # ── Face similarity ────────────────────────────────────────────────────
        gen_face_emb = self._get_face_emb(generated_image)
        ref_face_emb = self._get_face_emb(reference_image)
        face_detected = gen_face_emb is not None and ref_face_emb is not None

        if face_detected:
            face_sim = self._cosine(gen_face_emb, ref_face_emb)
        else:
            face_sim = 0.0
            print('  ⚠️  Face not detected in generated image')

        # ── Body similarity ────────────────────────────────────────────────────
        gen_body_emb = self._get_body_emb(generated_image)
        ref_body_emb = self._get_body_emb(reference_image)
        body_sim     = self._cosine(gen_body_emb, ref_body_emb)

        # ── Overall weighted score ─────────────────────────────────────────────
        if face_detected:
            overall = face_weight * face_sim + body_weight * body_sim
        else:
            overall = body_sim  # fall back to body only

        # ── Grade ──────────────────────────────────────────────────────────────
        grade = 'poor'
        for g, threshold in self.THRESHOLDS.items():
            if overall >= threshold:
                grade = g
                break

        return {
            'face_similarity' : round(face_sim, 4),
            'body_similarity' : round(body_sim, 4),
            'overall_score'   : round(overall, 4),
            'grade'           : grade,
            'face_detected'   : face_detected,
            'pass'            : overall >= self.THRESHOLDS['acceptable'],
        }

    def score_batch(self, generated_images: list, reference_image: Image.Image) -> list:
        """Score multiple generated images and sort by quality."""
        scores = [self.score(img, reference_image) for img in generated_images]
        return sorted(
            zip(generated_images, scores),
            key=lambda x: x[1]['overall_score'],
            reverse=True,
        )

print('IdentityConsistencyScorer defined')

---
## 👗 Step 6E — Garment Consistency Scorer

Separately scores how well the target outfit was reproduced in the output.

In [ ]:
class GarmentConsistencyScorer:
    """
    Scores how accurately the target garment was reproduced in the output.

    Uses two complementary signals:
        CLIP similarity  — semantic match (correct garment type and color)
        DINOv2 similarity — texture match (correct fabric/pattern details)
    """

    def __init__(self, clip_model, clip_preprocess, dino_model, dino_processor, device='cuda'):
        self.clip_model      = clip_model
        self.clip_preprocess = clip_preprocess
        self.dino_model      = dino_model
        self.dino_processor  = dino_processor
        self.device          = device
        print('✅ GarmentConsistencyScorer ready')

    @torch.no_grad()
    def _clip_emb(self, image: Image.Image) -> np.ndarray:
        x   = self.clip_preprocess(image).unsqueeze(0).to(self.device)
        emb = self.clip_model.encode_image(x)
        return F.normalize(emb, dim=-1).squeeze().cpu().numpy()

    @torch.no_grad()
    def _dino_emb(self, image: Image.Image) -> np.ndarray:
        inputs = self.dino_processor(images=image, return_tensors='pt').to(self.device)
        out    = self.dino_model(**inputs)
        emb    = F.normalize(out.last_hidden_state[:, 0, :], dim=-1)
        return emb.squeeze().cpu().numpy()

    @staticmethod
    def _cosine(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

    def score(
        self,
        generated_image  : Image.Image,
        reference_garment: Image.Image,
        clip_weight      : float = 0.6,
        texture_weight   : float = 0.4,
    ) -> dict:
        """
        Score garment transfer quality.

        Returns dict:
            'clip_similarity'    : float 0–1 (semantic match)
            'texture_similarity' : float 0–1 (pattern/fabric match)
            'overall_score'      : float 0–1 (weighted)
            'pass'               : bool (score > 0.45)
        """
        gen_clip = self._clip_emb(generated_image)
        ref_clip = self._clip_emb(reference_garment)
        clip_sim = self._cosine(gen_clip, ref_clip)

        gen_tex  = self._dino_emb(generated_image)
        ref_tex  = self._dino_emb(reference_garment)
        tex_sim  = self._cosine(gen_tex, ref_tex)

        overall  = clip_weight * clip_sim + texture_weight * tex_sim

        return {
            'clip_similarity'    : round(clip_sim, 4),
            'texture_similarity' : round(tex_sim, 4),
            'overall_score'      : round(overall, 4),
            'pass'               : overall >= 0.45,
        }

print('GarmentConsistencyScorer defined')

---
## 🩺 Step 6F — Anatomy & Quality Checker

Detects common diffusion artifacts before they reach the user:
- Extra/missing limbs (anatomy score)
- Motion blur / general blurriness (sharpness score)
- Skin tone uniformity (consistency score)
- Aspect ratio anomalies

In [ ]:
class QualityChecker:
    """
    Detects quality issues in generated images without needing ground truth.

    Checks:
        sharpness     — Laplacian variance (blurry images score low)
        skin_tone     — color consistency across detected face + body regions
        anatomy       — uses pose detector to count expected keypoints
        clip_quality  — CLIP aesthetic score proxy
    """

    def __init__(self, face_app, device='cuda'):
        self.face_app = face_app
        self.device   = device
        print('✅ QualityChecker ready')

    def sharpness_score(self, image: Image.Image) -> float:
        """
        Laplacian variance — measures image sharpness.
        Higher = sharper. Blurry images score < 50, sharp images > 200.
        Normalized to 0–1 range (capped at variance=500).
        """
        gray    = cv2.cvtColor(np.array(image.convert('RGB')), cv2.COLOR_RGB2GRAY)
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        return float(min(lap_var / 500.0, 1.0))

    def face_quality_score(self, image: Image.Image) -> dict:
        """
        InsightFace provides a det_score (detection confidence) for each face.
        High det_score = clear, well-formed face.
        Returns average score across detected faces.
        """
        img_bgr = cv2.cvtColor(np.array(image.convert('RGB')), cv2.COLOR_RGB2BGR)
        faces   = self.face_app.get(img_bgr)

        if not faces:
            return {'face_quality': 0.0, 'face_count': 0, 'face_detected': False}

        avg_quality = float(np.mean([f.det_score for f in faces]))
        return {
            'face_quality'  : round(avg_quality, 4),
            'face_count'    : len(faces),
            'face_detected' : True,
        }

    def color_consistency_score(self, image: Image.Image) -> float:
        """
        Checks for unnatural color banding or extreme saturation.
        Converts to HSV and measures hue/saturation variance.
        Realistic images have moderate, consistent saturation.
        """
        img_rgb = np.array(image.convert('RGB'))
        img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)

        sat_mean = img_hsv[:, :, 1].mean() / 255.0   # avg saturation
        sat_std  = img_hsv[:, :, 1].std()  / 255.0   # saturation variance

        # Ideal: moderate saturation (0.2–0.6), not over-saturated or washed out
        sat_score = 1.0 - abs(sat_mean - 0.35) * 2.0  # peaks at sat=0.35
        var_score = 1.0 - min(sat_std * 3.0, 1.0)     # lower variance = more consistent

        return float(max(0.0, (sat_score + var_score) / 2.0))

    def check(
        self,
        image      : Image.Image,
        min_sharp  : float = 0.15,
        min_face_q : float = 0.60,
    ) -> dict:
        """
        Run all quality checks and produce a pass/fail verdict.

        Returns dict:
            'sharpness'          : float 0–1
            'face_quality'       : float 0–1
            'color_consistency'  : float 0–1
            'overall_quality'    : float 0–1
            'issues'             : list of str (identified problems)
            'pass'               : bool
        """
        issues = []

        sharpness   = self.sharpness_score(image)
        face_info   = self.face_quality_score(image)
        color_score = self.color_consistency_score(image)

        face_q = face_info.get('face_quality', 0.0)

        if sharpness < min_sharp:
            issues.append(f'blurry image (sharpness={sharpness:.2f})')
        if not face_info['face_detected']:
            issues.append('no face detected in output')
        elif face_q < min_face_q:
            issues.append(f'low face quality score ({face_q:.2f})')
        if color_score < 0.3:
            issues.append(f'color inconsistency detected ({color_score:.2f})')

        overall = np.mean([sharpness, face_q, color_score])

        return {
            'sharpness'         : round(sharpness,   4),
            'face_quality'      : round(face_q,      4),
            'color_consistency' : round(color_score, 4),
            'overall_quality'   : round(float(overall), 4),
            'issues'            : issues,
            'pass'              : len(issues) == 0,
        }

print('QualityChecker defined')

---
## 🔁 Step 6G — Auto-Retry Engine

If a generation fails any quality gate, the engine automatically adjusts
generation parameters and retries — up to `max_retries` times.

In [ ]:
@dataclass
class GenerationParams:
    """Parameters for one generation attempt."""
    prompt           : str
    negative_prompt  : str  = ''
    num_steps        : int  = 30
    guidance_scale   : float = 7.5
    controlnet_scale : float = 0.85
    identity_scale   : float = 0.80
    garment_scale    : float = 0.70
    seed             : int  = 42


class AutoRetryEngine:
    """
    Automatically retries generation when quality gates fail.

    Retry strategy per failure type:
        blurry image        → increase num_steps, increase guidance_scale
        low face quality    → increase identity_scale, increase num_steps
        garment mismatch    → increase garment_scale
        poor identity match → increase identity_scale, change seed
        repeated failure    → increase controlnet_scale (tighter pose)
    """

    def __init__(
        self,
        fusion_pipeline,
        post_processor,
        max_retries         : int   = 3,
        identity_threshold  : float = 0.45,
        garment_threshold   : float = 0.45,
        quality_threshold   : float = 0.40,
    ):
        self.fusion          = fusion_pipeline
        self.pp              = post_processor
        self.max_retries     = max_retries
        self.id_threshold    = identity_threshold
        self.gar_threshold   = garment_threshold
        self.qual_threshold  = quality_threshold
        print('✅ AutoRetryEngine ready')

    def _adjust_params(self, params: GenerationParams, issues: list, attempt: int) -> GenerationParams:
        """Modify generation parameters based on what went wrong."""
        p = GenerationParams(**vars(params))  # copy
        p.seed = params.seed + attempt * 7    # always change seed on retry

        for issue in issues:
            if 'blurry' in issue:
                p.num_steps      = min(p.num_steps + 5, 50)
                p.guidance_scale = min(p.guidance_scale + 0.5, 12.0)

            if 'face' in issue:
                p.identity_scale = min(p.identity_scale + 0.1, 1.0)
                p.num_steps      = min(p.num_steps + 5, 50)

            if 'garment' in issue or 'outfit' in issue:
                p.garment_scale  = min(p.garment_scale + 0.1, 1.0)

            if 'identity' in issue:
                p.identity_scale = min(p.identity_scale + 0.15, 1.0)

        # On final attempt: max everything out
        if attempt >= self.max_retries - 1:
            p.num_steps      = 45
            p.identity_scale = min(params.identity_scale + 0.2, 1.0)
            p.garment_scale  = min(params.garment_scale  + 0.15, 1.0)

        return p

    def generate_with_retry(
        self,
        person_image  : Image.Image,
        pose_image    : Image.Image,
        garment_image : Image.Image,
        params        : GenerationParams,
        verbose       : bool = True,
    ) -> dict:
        """
        Generate with automatic retry on quality failure.

        Returns dict:
            'image'           : PIL Image — best result
            'attempt'         : int       — which attempt succeeded
            'scores'          : dict      — final quality scores
            'all_attempts'    : list      — (image, scores) for every attempt
            'success'         : bool
        """
        all_attempts = []
        current_params = params

        for attempt in range(self.max_retries + 1):
            if verbose:
                print(f'\n  Attempt {attempt + 1}/{self.max_retries + 1}')
                print(f'    steps={current_params.num_steps}  '
                      f'id_scale={current_params.identity_scale:.2f}  '
                      f'gar_scale={current_params.garment_scale:.2f}  '
                      f'seed={current_params.seed}')

            # ── Generate ───────────────────────────────────────────────────
            raw_result = self.fusion.generate(
                person_image     = person_image,
                pose_image       = pose_image,
                garment_image    = garment_image,
                prompt           = current_params.prompt,
                negative_prompt  = current_params.negative_prompt,
                num_steps        = current_params.num_steps,
                guidance_scale   = current_params.guidance_scale,
                controlnet_scale = current_params.controlnet_scale,
                identity_scale   = current_params.identity_scale,
                garment_scale    = current_params.garment_scale,
                seed             = current_params.seed,
            )
            raw_image = raw_result['image']

            # ── Post-process ───────────────────────────────────────────────
            processed = self.pp.process(raw_image, person_image, garment_image)

            # ── Score ──────────────────────────────────────────────────────
            scores = processed['scores']
            all_attempts.append((processed['final_image'], scores, current_params))

            if verbose:
                print(f'    identity={scores["identity"]["overall_score"]:.3f}  '
                      f'garment={scores["garment"]["overall_score"]:.3f}  '
                      f'quality={scores["quality"]["overall_quality"]:.3f}')

            # ── Check pass ─────────────────────────────────────────────────
            id_pass  = scores['identity']['overall_score'] >= self.id_threshold
            gar_pass = scores['garment']['overall_score']  >= self.gar_threshold
            qua_pass = scores['quality']['overall_quality'] >= self.qual_threshold

            if id_pass and gar_pass and qua_pass:
                if verbose:
                    print(f'    ✅ Passed all quality gates on attempt {attempt + 1}')
                return {
                    'image'       : processed['final_image'],
                    'attempt'     : attempt + 1,
                    'scores'      : scores,
                    'all_attempts': all_attempts,
                    'success'     : True,
                }

            # ── Build issue list for next retry ────────────────────────────
            retry_issues = []
            if not id_pass  : retry_issues.append('low identity match')
            if not gar_pass : retry_issues.append('garment mismatch')
            if not qua_pass : retry_issues.append(str(scores['quality']['issues']))

            if attempt < self.max_retries:
                current_params = self._adjust_params(current_params, retry_issues, attempt + 1)
                if verbose:
                    print(f'    ⚠️  Issues: {retry_issues} — adjusting params...')

        # ── All retries exhausted: return best attempt ─────────────────────
        if verbose:
            print('\n  ⚠️  Max retries reached — returning best attempt')

        best = max(all_attempts, key=lambda x: x[1]['identity']['overall_score'])
        return {
            'image'       : best[0],
            'attempt'     : self.max_retries + 1,
            'scores'      : best[1],
            'all_attempts': all_attempts,
            'success'     : False,
        }

print('AutoRetryEngine defined')

---
## 🔍 Step 6H — Real-ESRGAN Upscaler

In [ ]:
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

class ImageUpscaler:
    """
    Upscales images using Real-ESRGAN (2× or 4×).

    Real-ESRGAN is specifically trained on photorealistic content —
    it preserves skin texture, fabric patterns, and hair detail
    much better than bicubic or Lanczos interpolation.

    Model options:
        RealESRGAN_x2plus  — 2× upscale, fast (~0.5s on T4)
        RealESRGAN_x4plus  — 4× upscale, slower (~1.5s on T4)
    """

    MODEL_URLS = {
        2: 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
        4: 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    }

    def __init__(self, scale: int = 2, device: str = 'cuda'):
        assert scale in (2, 4), 'scale must be 2 or 4'
        self.scale  = scale
        model_path  = f'RealESRGAN_x{scale}plus.pth'

        # Download weights if needed
        if not Path(model_path).exists():
            import urllib.request
            print(f'  Downloading Real-ESRGAN x{scale} weights...')
            urllib.request.urlretrieve(self.MODEL_URLS[scale], model_path)

        # Model architecture varies by scale
        num_block = 6 if scale == 2 else 23
        model = RRDBNet(
            num_in_ch=3, num_out_ch=3,
            num_feat=64, num_block=num_block, num_grow_ch=32,
            scale=scale,
        )
        self.upsampler = RealESRGANer(
            scale      = scale,
            model_path = model_path,
            model      = model,
            tile       = 512,        # tile size for memory efficiency
            tile_pad   = 10,
            pre_pad    = 0,
            half       = True if device == 'cuda' else False,
        )
        print(f'✅ ImageUpscaler ready (Real-ESRGAN x{scale})')

    def upscale(self, image: Image.Image) -> Image.Image:
        """
        Upscale a PIL image by self.scale factor.
        Input:  768×1024 → Output: 1536×2048 (x2) or 3072×4096 (x4)
        """
        img_bgr     = cv2.cvtColor(np.array(image.convert('RGB')), cv2.COLOR_RGB2BGR)
        upscaled, _ = self.upsampler.enhance(img_bgr, outscale=self.scale)
        return Image.fromarray(cv2.cvtColor(upscaled, cv2.COLOR_BGR2RGB))


upscaler = ImageUpscaler(scale=2, device=device)

---
## 🎛️ Step 6I — PostProcessor: The Master Class

One class that wraps face restoration → scoring → upscaling in the right order.

In [ ]:
class PostProcessor:
    """
    Master post-processing pipeline.

    Given a raw generated image + reference inputs, this:
        1. Restores face detail (GFPGAN)
        2. Scores identity consistency (ArcFace)
        3. Scores garment consistency (CLIP + DINOv2)
        4. Runs anatomy/quality checks
        5. Applies final sharpening + color correction
        6. Upscales to final resolution (Real-ESRGAN x2)

    Returns the polished final image + a full quality report.
    """

    def __init__(
        self,
        face_restorer,
        identity_scorer,
        garment_scorer,
        quality_checker,
        upscaler,
        do_upscale    : bool  = True,
        fidelity_weight: float = 0.5,
    ):
        self.face_restorer    = face_restorer
        self.identity_scorer  = identity_scorer
        self.garment_scorer   = garment_scorer
        self.quality_checker  = quality_checker
        self.upscaler         = upscaler
        self.do_upscale       = do_upscale
        self.fidelity_weight  = fidelity_weight
        print('✅ PostProcessor assembled')

    def process(
        self,
        generated_image  : Image.Image,
        reference_person : Image.Image,
        reference_garment: Image.Image,
    ) -> dict:
        """
        Full post-processing pass.

        Returns dict:
            'raw_image'      : PIL Image — original diffusion output
            'restored_image' : PIL Image — after face restoration
            'final_image'    : PIL Image — after upscaling
            'scores'         : dict      — identity, garment, quality scores
            'report'         : str       — human-readable quality report
        """
        results = {'raw_image': generated_image}

        # ── Step 1: Face Restoration ───────────────────────────────────────────
        print('  [1/4] Restoring face detail (GFPGAN)...')
        restored = self.face_restorer.restore_face_only(
            generated_image,
            blend_alpha     = 0.85,
            fidelity_weight = self.fidelity_weight,
        )
        results['restored_image'] = restored

        # ── Step 2: Scoring ────────────────────────────────────────────────────
        print('  [2/4] Scoring consistency...')
        id_scores  = self.identity_scorer.score(restored, reference_person)
        gar_scores = self.garment_scorer.score(restored, reference_garment)
        qua_scores = self.quality_checker.check(restored)

        results['scores'] = {
            'identity': id_scores,
            'garment' : gar_scores,
            'quality' : qua_scores,
        }

        # ── Step 3: Final sharpening + color correction ────────────────────────
        print('  [3/4] Applying final sharpening...')
        sharpened = self._sharpen(restored, strength=1.3)

        # ── Step 4: Upscale ────────────────────────────────────────────────────
        if self.do_upscale:
            print('  [4/4] Upscaling 2× (Real-ESRGAN)...')
            final = self.upscaler.upscale(sharpened)
        else:
            final = sharpened
        results['final_image'] = final

        # ── Build human-readable report ────────────────────────────────────────
        results['report'] = self._build_report(results['scores'])

        return results

    def _sharpen(self, image: Image.Image, strength: float = 1.3) -> Image.Image:
        """Apply unsharp mask to bring out fine detail."""
        enhancer = ImageEnhance.Sharpness(image)
        return enhancer.enhance(strength)

    def _build_report(self, scores: dict) -> str:
        id_s   = scores['identity']
        gar_s  = scores['garment']
        qual_s = scores['quality']

        grade_emoji = {'excellent': '🌟', 'good': '✅', 'acceptable': '⚠️', 'poor': '❌'}
        grade = id_s.get('grade', 'poor')

        lines = [
            '═══════════════════════════════════════',
            '  GENERATION QUALITY REPORT',
            '═══════════════════════════════════════',
            f'  Identity   {grade_emoji.get(grade, "")} {grade.upper()}',
            f'    Face similarity : {id_s["face_similarity"]:.3f}',
            f'    Body similarity : {id_s["body_similarity"]:.3f}',
            f'    Overall         : {id_s["overall_score"]:.3f}',
            '',
            f'  Garment  {"✅" if gar_s["pass"] else "⚠️"}',
            f'    CLIP similarity    : {gar_s["clip_similarity"]:.3f}',
            f'    Texture similarity : {gar_s["texture_similarity"]:.3f}',
            f'    Overall            : {gar_s["overall_score"]:.3f}',
            '',
            f'  Quality  {"✅" if qual_s["pass"] else "⚠️"}',
            f'    Sharpness          : {qual_s["sharpness"]:.3f}',
            f'    Face quality       : {qual_s["face_quality"]:.3f}',
            f'    Color consistency  : {qual_s["color_consistency"]:.3f}',
        ]
        if qual_s['issues']:
            lines.append(f'    Issues : {", ".join(qual_s["issues"])}')
        lines.append('═══════════════════════════════════════')
        return '\n'.join(lines)


# ── Instantiate all scorers ─────────────────────────────────────────────────────
from transformers import AutoImageProcessor, AutoModel

print('Loading models for scoring...')
face_app_scorer = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider'] if device == 'cuda' else ['CPUExecutionProvider'],
)
face_app_scorer.prepare(ctx_id=0 if device == 'cuda' else -1, det_size=(640, 640))

dino_proc  = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
dino_mod   = AutoModel.from_pretrained('facebook/dinov2-base').to(device)
dino_mod.eval()

clip_mod, _, clip_prep = open_clip.create_model_and_transforms(
    'ViT-H-14', pretrained='laion2b_s32b_b79k', device=device
)
clip_mod.eval()

id_scorer  = IdentityConsistencyScorer(face_app_scorer, dino_mod, dino_proc, device)
gar_scorer = GarmentConsistencyScorer(clip_mod, clip_prep, dino_mod, dino_proc, device)
qual_check = QualityChecker(face_app_scorer, device)

post_processor = PostProcessor(
    face_restorer  = face_restorer,
    identity_scorer= id_scorer,
    garment_scorer = gar_scorer,
    quality_checker= qual_check,
    upscaler       = upscaler,
    do_upscale     = True,
    fidelity_weight= 0.5,
)

print('\n✅ Full PostProcessor ready!')

---
## 🧪 Step 6J — Full End-to-End Test

In [ ]:
from PIL import ImageDraw

# ── Load or create test images ─────────────────────────────────────────────────
def make_placeholder(color=(180,180,180), size=(768,1024)):
    img = Image.new('RGB', size, color)
    d = ImageDraw.Draw(img)
    cx, cy = size[0]//2, size[1]//2
    d.ellipse([cx-60, cy//3-60, cx+60, cy//3+60], fill=(220,190,160))
    d.rectangle([cx-50, cy//3+55, cx+50, cy//3+200], fill=(100,140,200))
    return img

# Try loading saved outputs from Step 5; fall back to placeholders
try:
    raw_output = Image.open('fusion_result_1.png').convert('RGB')
    print('✓ Loaded fusion_result_1.png from Step 5')
except FileNotFoundError:
    raw_output = make_placeholder((200, 210, 220))
    print('ℹ️  Using placeholder — run Step 5 first for real output')

try:
    ref_person = Image.open('my_reference.jpg').convert('RGB')
    print('✓ Loaded reference person')
except FileNotFoundError:
    ref_person = make_placeholder((210, 190, 170))

try:
    ref_garment = Image.open('my_garment_segmented.png').convert('RGB')
    print('✓ Loaded reference garment')
except FileNotFoundError:
    ref_garment = Image.new('RGB', (512, 512), (60, 100, 180))

print(f'\nInput image size: {raw_output.size}')

In [ ]:
# ── Run full post-processing ───────────────────────────────────────────────────
print('Running post-processor...\n')
pp_result = post_processor.process(
    generated_image   = raw_output,
    reference_person  = ref_person,
    reference_garment = ref_garment,
)

print('\n' + pp_result['report'])

In [ ]:
# ── Visualize the pipeline stages ─────────────────────────────────────────────
stages = [
    ('Raw output\n(from Step 5)', pp_result['raw_image']),
    ('Face restored\n(GFPGAN)',   pp_result['restored_image']),
    ('Final output\n(2× upscaled)', pp_result['final_image'].resize(
        pp_result['raw_image'].size, Image.LANCZOS)  # resize back for display
    ),
]

fig, axes = plt.subplots(1, len(stages), figsize=(6 * len(stages), 9))
fig.suptitle('Post-Processing Pipeline Stages', fontsize=14, fontweight='bold')

scores = pp_result['scores']

for i, (title, img) in enumerate(stages):
    axes[i].imshow(img)
    axes[i].set_title(title, fontsize=10)
    axes[i].axis('off')

    if i == 2:  # final image — add score overlay
        id_score  = scores['identity']['overall_score']
        gar_score = scores['garment']['overall_score']
        qual_score= scores['quality']['overall_quality']
        grade     = scores['identity'].get('grade', 'n/a')
        axes[i].set_title(
            f'Final output (2× upscaled)\n'
            f'Identity: {id_score:.2f} [{grade}]  '
            f'Garment: {gar_score:.2f}  '
            f'Quality: {qual_score:.2f}',
            fontsize=9,
        )

plt.tight_layout()
plt.savefig('postprocessing_stages.png', dpi=120, bbox_inches='tight')
plt.show()

# Save final image
pp_result['final_image'].save('final_output.png')
print('💾 Final output saved → final_output.png')
print(f'   Resolution: {pp_result["final_image"].size}')

In [ ]:
# ── Score bar chart ────────────────────────────────────────────────────────────
labels = [
    'Face similarity', 'Body similarity', 'Identity overall',
    'Garment CLIP', 'Garment texture', 'Garment overall',
    'Sharpness', 'Face quality', 'Color consistency',
]
values = [
    scores['identity']['face_similarity'],
    scores['identity']['body_similarity'],
    scores['identity']['overall_score'],
    scores['garment']['clip_similarity'],
    scores['garment']['texture_similarity'],
    scores['garment']['overall_score'],
    scores['quality']['sharpness'],
    scores['quality']['face_quality'],
    scores['quality']['color_consistency'],
]
colors = [
    '#7F77DD','#7F77DD','#534AB7',
    '#D85A30','#D85A30','#993C1D',
    '#1D9E75','#1D9E75','#0F6E56',
]
thresholds = [0.7,0.5,0.45, 0.6,0.4,0.45, 0.15,0.60,0.30]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(labels, values, color=colors, alpha=0.85)

# Threshold lines
for i, (val, thresh) in enumerate(zip(values, thresholds)):
    ax.plot([thresh, thresh], [i - 0.4, i + 0.4], 'k--', alpha=0.4, lw=1)
    ax.text(val + 0.01, i, f'{val:.3f}', va='center', fontsize=9)

ax.set_xlim(0, 1.1)
ax.set_xlabel('Score (0–1)')
ax.set_title('Quality Scores — dashed lines = pass thresholds', fontsize=12, fontweight='bold')
ax.axvline(0, color='gray', lw=0.5)

plt.tight_layout()
plt.savefig('quality_scores.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 💾 Step 6K — Save Everything

In [ ]:
import json

# Load and update pipeline config
try:
    with open('pipeline_config.json') as f:
        config = json.load(f)
except FileNotFoundError:
    config = {}

config.update({
    'postprocessing': {
        'face_restorer'         : 'GFPGANv1.3',
        'face_fidelity_weight'  : 0.5,
        'face_blend_alpha'      : 0.85,
        'upscaler'              : 'RealESRGAN_x2plus',
        'sharpening_strength'   : 1.3,
        'thresholds': {
            'identity_min'  : 0.45,
            'garment_min'   : 0.45,
            'quality_min'   : 0.40,
        },
    },
    'final_output_size': [1536, 2048],
})

with open('pipeline_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Step 6 complete!')
print()
print('Saved files:')
print('  final_output.png             ← polished 1536×2048 output')
print('  postprocessing_stages.png    ← before/after comparison')
print('  quality_scores.png           ← score bar chart')
print('  pipeline_config.json         ← updated full config')
print()
print('Next → Step 7: Gradio Web UI — tie everything into a live interactive app!')

---
## 📋 Summary

| Component | Model | What it fixes |
|-----------|-------|---------------|
| `FaceRestorer` | GFPGAN v1.3 | Blurry / soft faces from diffusion |
| `IdentityConsistencyScorer` | ArcFace + DINOv2 | Measures face/body match vs reference |
| `GarmentConsistencyScorer` | CLIP + DINOv2 | Measures outfit transfer accuracy |
| `QualityChecker` | Laplacian + InsightFace | Detects blur, bad anatomy, color issues |
| `AutoRetryEngine` | — | Rejects failures, auto-adjusts, retries |
| `ImageUpscaler` | Real-ESRGAN x2 | 768×1024 → 1536×2048 sharp output |
| `PostProcessor` | All of the above | Single `.process()` call |

## Complete project status

```
Step 1+2  ✅  Identity encoder      (face + body embedding)
Step 3    ✅  Pose control           (ControlNet + DWPose)
Step 4    ✅  Garment encoder        (CLIP + DINOv2 + projector)
Step 5    ✅  Fusion pipeline        (identity + pose + garment → generation)
Step 6    ✅  Post-processing        (restoration + scoring + upscaling)
Step 7    ⬅️  Gradio Web UI          (interactive app — final step!)
```